#Worksheet 6

Transfer Learning

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
os.listdir("/content/drive/MyDrive/AI ML 2026/FruitinAmazon/test")

['cupuacu', 'acai', 'pupunha', 'graviola', 'tucuma', 'guarana']

In [24]:
pip install keras tensorflow

In [25]:
import tensorflow as tf
print(tf.keras.__version__)

3.13.2


In [26]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = train_datagen.flow_from_directory(
    "/content/drive/MyDrive/AI ML 2026/FruitinAmazon/train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    "/content/drive/MyDrive/AI ML 2026/FruitinAmazon/train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

test_datagen = ImageDataGenerator(rescale=1./255)

test_data = test_datagen.flow_from_directory(
    "/content/drive/MyDrive/AI ML 2026/FruitinAmazon/test",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

Found 72 images belonging to 6 classes.
Found 18 images belonging to 6 classes.
Found 30 images belonging to 6 classes.


In [27]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze layers
for layer in base_model.layers:
    layer.trainable = False

# Custom head
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(train_data.num_classes, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=output)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [28]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 59s 17s/step - accuracy: 0.2222 - loss: 2.1920 - val_accuracy: 0.4444 - val_loss: 1.4400
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step - accuracy: 0.5278 - loss: 1.2636 - val_accuracy: 0.7778 - val_loss: 0.9764
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step - accuracy: 0.7361 - loss: 0.7855 - val_accuracy: 0.8889 - val_loss: 0.7294
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 194ms/step - accuracy: 0.7778 - loss: 0.6576 - val_accuracy: 0.9444 - val_loss: 0.5040
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 178ms/step - accuracy: 0.9167 - loss: 0.3900 - val_accuracy: 1.0000 - val_loss: 0.3710
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 189ms/step - accuracy: 0.9444 - loss: 0.2993 - val_accuracy: 0.9444 - val_loss: 0.3337
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 485ms/step - accuracy: 0.9861 - loss: 0.1627 - val_accuracy: 0.9444 - val_loss: 0.3028
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 417ms/step - accuracy: 0.9861 - loss: 0.1213 - val_accuracy: 1.0000 - val_loss: 

In [29]:
loss, acc = model.evaluate(test_data)
print("Test Accuracy:", acc)


1/1 ━━━━━━━━━━━━━━━━━━━━ 12s 12s/step - accuracy: 0.9667 - loss: 0.2688
Test Accuracy: 0.9666666388511658


In [30]:
from sklearn.metrics import classification_report
import numpy as np

predictions = model.predict(test_data)
y_pred = np.argmax(predictions, axis=1)

print(classification_report(
    test_data.classes,
    y_pred,
    target_names=list(test_data.class_indices.keys())
))

1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step
              precision    recall  f1-score   support

        acai       1.00      1.00      1.00         5
     cupuacu       1.00      1.00      1.00         5
    graviola       1.00      1.00      1.00         5
     guarana       1.00      1.00      1.00         5
     pupunha       0.83      1.00      0.91         5
      tucuma       1.00      0.80      0.89         5

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30



In [33]:
from tensorflow.keras.preprocessing import image
import numpy as np

img_path = "/content/drive/MyDrive/AI ML 2026/FruitinAmazon/train/pupunha/images (1).jpeg"  # change this

img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

pred = model.predict(img_array)
pred_class = list(train_data.class_indices.keys())[np.argmax(pred)]

print("Prediction:", pred_class)

1/1 ━━━━━━━━━━━━━━━━━━━━ 13s 13s/step
Prediction: pupunha


In [34]:
from tensorflow.keras import Sequential

scratch_model = Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(train_data.num_classes, activation='softmax')
])

scratch_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

scratch_model.fit(train_data, validation_data=val_data, epochs=10)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 9s 3s/step - accuracy: 0.1806 - loss: 11.1461 - val_accuracy: 0.2222 - val_loss: 7.9091
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step - accuracy: 0.2778 - loss: 5.0237 - val_accuracy: 0.2222 - val_loss: 2.1069
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step - accuracy: 0.3194 - loss: 1.6620 - val_accuracy: 0.3333 - val_loss: 1.5299
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step - accuracy: 0.5417 - loss: 1.2508 - val_accuracy: 0.5000 - val_loss: 1.2766
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step - accuracy: 0.7917 - loss: 0.8450 - val_accuracy: 0.5000 - val_loss: 1.5776
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step - accuracy: 0.8333 - loss: 0.6351 - val_accuracy: 0.5000 - val_loss: 1.3054
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 197ms/step - accuracy: 0.9167 - loss: 0.3299 - val_accuracy: 0.6667 - val_loss: 0.9840
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step - accuracy: 0.9444 - loss: 0.1917 - val_accuracy: 0.6111 - val_loss: 1

In [35]:
predictions = scratch_model.predict(test_data)
y_pred = np.argmax(predictions, axis=1)

print(classification_report(
    test_data.classes,
    y_pred,
    target_names=list(test_data.class_indices.keys())
))


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
              precision    recall  f1-score   support

        acai       0.42      1.00      0.59         5
     cupuacu       1.00      0.40      0.57         5
    graviola       0.40      0.40      0.40         5
     guarana       1.00      0.80      0.89         5
     pupunha       1.00      1.00      1.00         5
      tucuma       1.00      0.40      0.57         5

    accuracy                           0.67        30
   macro avg       0.80      0.67      0.67        30
weighted avg       0.80      0.67      0.67        30



In [37]:
from tensorflow.keras.preprocessing import image
import numpy as np

img_path = "/content/drive/MyDrive/AI ML 2026/FruitinAmazon/train/pupunha/images (1).jpeg"  # change this

img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

pred = scratch_model.predict(img_array)
pred_class = list(train_data.class_indices.keys())[np.argmax(pred)]

print("Prediction:", pred_class)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Prediction: pupunha
